---
title: Handle Categorical Variables in LightGBM
created: '2022-03-26T17:37:54-07:00'
date: '2026-11-22T17:33:27-08:00'
authors:
  - bendu
label: handle-categorical-variables-in-lightgbm
license: CC-BY-4.0
tags:
  - computer science
  - programming
  - AI
  - data science
  - machine learning
  - LightGBM
  - categorical
  - category
---

**Things on this page are fragmentary and immature notes/thoughts of the author. Please read with your own judgement!**


LightGBM support pandas columns of `category` type. 
As a matter of fact,
this is the suggested way of handling categorical columns in LightGBM.

    data[feature] = pd.Series(data[feature], dtype="category")

A LightGBM model (which is a Booster object) records categories of each categorical feature. 
This information is used to set categories of each categorical feature during prediction,
which ensures that a LightGBM model can always handle categorical features correctly.

However, 
be careful about importance of categorical features.
The article 
[Beware of categorical features in LGBM!](https://www.kaggle.com/code/mlisovyi/beware-of-categorical-features-in-lgbm/notebook)
argues that the Python library shap is better (than built-in methods) 
for reporting feature importance,
especially when you deal with categorical features. 
Please refer to 
[Interpreting a LightGBM model](https://www.kaggle.com/code/slundberg/interpreting-a-lightgbm-model/notebook)
for more details on how to use shap.

## Categorical Variables with Missing Values

LightGBM represents a categorical feature as non-negative integers (category codes).
Missing values are handled as below.

1. When a pandas column of the `category` type is passed to LightGBM,
   pandas encodes missing values (`NaN`) as the category code `-1`,
   which LightGBM treats as missing.
   As a matter of fact,
   LightGBM treats **all negative values** (and `NaN`) in a categorical feature as missing values.
   So, you do NOT have to do anything special for missing values
   if you use the `category` type in pandas.

2. If you encode categorical features as integers manually,
   encode valid categories as contiguous non-negative integers starting from 0
   (values are cast to int32 and must be less than `Int32.MaxValue`)
   and encode missing values as `NaN` or a negative integer (e.g., `-1`).
   Never encode missing values as a valid category code (e.g., `0`)
   unless you really want missing values to be treated as a regular category.
   Notice that zeros are also treated as missing values if you set `zero_as_missing=true`,
   which collides with the category code 0.
   Floating point values in categorical features are rounded towards 0.

3. When splitting a categorical feature,
   LightGBM sends the selected categories to the left child
   and everything else (including missing values) to the right child.
   Categories unseen during training are treated as missing values at prediction time.
   Missing value handling is enabled by default
   and can be disabled by setting `use_missing=false`.

4. If missing values are informative,
   an alternative is to explicitly treat them as a separate category
   (e.g., by adding a category `"missing"` to the pandas column and filling `NaN` with it).

Other gradient boosting libraries handle categorical variables with missing values differently.

- **XGBoost**: Native categorical support
  (`enable_categorical=True` with pandas `category` columns
  and `tree_method="hist"` (default since 2.0) or `"approx"`)
  treats `NaN` as missing values
  and learns a default direction for missing values at each split,
  the same way as for numerical features.
  Invalid category codes (negative or not less than the number of categories)
  are rejected during training.

- **CatBoost**: Categorical features must be integers or strings and cannot contain `NaN`.
  You have to fill missing values with a placeholder (e.g., `"missing"`) before training,
  which makes missing values a separate category.

## Binary Response Variables

For `objective="binary"`,
the LightGBM documentation says labels should be in {0, 1}.
In practice,
the binary objective treats labels > 0 as positive and labels <= 0 as negative,
so 1/-1 labels do not raise an error.
Still, encode labels as 1/0 to match the documentation and to avoid confusion with other tools.

## References 

https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html

[LightGBM FAQ](https://lightgbm.readthedocs.io/en/latest/FAQ.html#general-lightgbm-questions)

[LightGBM Advanced Topics](https://lightgbm.readthedocs.io/en/latest/Advanced-Topics.html)

[XGBoost - Categorical Data](https://xgboost.readthedocs.io/en/stable/tutorials/categorical.html)

[CatBoost - Categorical features](https://catboost.ai/docs/en/features/categorical-features)